In [1]:
import pandas as pd
import json
import geopandas as gpd
from shapely import wkt

In [2]:
wijk = pd.read_csv("C:/Users/isamu/OneDrive/THESIS/DATA/GEBIED_WIJKEN.csv", delimiter = ";")
hotelbeds = pd.read_csv("C:/Users/isamu/OneDrive/THESIS/DATA/Amsterdam/hotelbed_wijk.csv", delimiter = ";")

In [3]:
hotels_rev = pd.read_csv("C:/Users/isamu/OneDrive/THESIS/DATA/Google/P2 Data/2024_count.csv")
hotels_loc = pd.read_csv("C:/Users/isamu/OneDrive/THESIS/DATA/Google/P2 Data/times.csv")[["place_id", "coordinates", "main_category"]]

In [4]:
hotelbeds["Wijkcode"] = hotelbeds["Wijken"].str.split().str[0]
hotelbeds["Wijknaam"] = hotelbeds["Wijken"].str.extract(r'^\S+\s+(.*)')


In [5]:
wijk = wijk.merge(hotelbeds, on = "Wijkcode")
wijk['geometry'] = wijk['WKT_LNG_LAT'].apply(wkt.loads)
wijk_gdf = gpd.GeoDataFrame(wijk, geometry = "geometry", crs = "EPSG:4326")

In [6]:
# Parse the 'coordinates' column into separate latitude and longitude columns
hotels_loc[['latitude', 'longitude']] = hotels_loc['coordinates'].apply(lambda x: pd.Series(json.loads(x)))

hotels_loc = hotels_loc[hotels_loc["main_category"] == "Hotel"]

In [7]:
hotels_gdf = hotels_rev.merge(hotels_loc, on = "place_id", how = "inner")

hotels_gdf = gpd.GeoDataFrame(hotels_gdf, geometry=gpd.points_from_xy(hotels_gdf['longitude'], hotels_gdf['latitude']), crs="EPSG:4326")

In [8]:
hotels_wijk_gdf = gpd.sjoin(hotels_gdf, wijk_gdf, how = "inner", predicate = "intersects")
hotels_wijk_gdf = hotels_wijk_gdf[["place_id", "reviews 2024", "name", "geometry", "Wijkcode", "Wijknaam_x", "Aantal hotelbedden"]]

In [9]:
hotels_wijk_gdf_sum = hotels_wijk_gdf.groupby(["Wijkcode", "Wijknaam_x", "Aantal hotelbedden"])["reviews 2024"].sum().reset_index()

In [10]:
hotels_wijk_gdf = hotels_wijk_gdf.merge(hotels_wijk_gdf_sum, on = ["Wijkcode", "Wijknaam_x", "Aantal hotelbedden"], suffixes = ("","_sum"))
hotels_wijk_gdf["percentage_reviews_wijk"] = hotels_wijk_gdf["reviews 2024"] / hotels_wijk_gdf["reviews 2024_sum"]
hotels_wijk_gdf["hotelbeds hotel"] = round(hotels_wijk_gdf["percentage_reviews_wijk"] * hotels_wijk_gdf["Aantal hotelbedden"])
hotels_wijk_gdf

,place_id,reviews 2024,name,geometry,Wijkcode,Wijknaam_x,Aantal hotelbedden,reviews 2024_sum,percentage_reviews_wijk,hotelbeds hotel
0,ChIJbeLDaSIKxkcR4uMOfG8J_cw,784,Hotel nhow Amsterdam RAI,POINT (4.89191 52.33896),K23,Zuidas,3129,1989,0.394168,1233.0
1,ChIJZyc7C7oJxkcRSPqsBgyVa0k,760,DoubleTree by Hilton Amsterdam Centraal Station,POINT (4.90547 52.37668),A04,Nieuwmarkt/Lastage,3574,2191,0.346874,1240.0
2,ChIJgYCiu5kLxkcRPQIZ1Wezozs,722,"Holiday Inn Express Amsterdam - Arena Towers, ...",POINT (4.94102 52.3096),T92,Amstel III/Bullewijk,4695,3196,0.225907,1061.0
3,ChIJxYYOH_bixUcRmvXuhmyjUx4,709,MEININGER Hotel Amsterdam City West,POINT (4.83746 52.39036),F11,Bedrijventerrein Sloterdijk,4705,2963,0.239285,1126.0
4,ChIJhfOaePYJxkcRWWJAZeXdlp4,637,Leonardo Hotel Amsterdam Rembrandtpark,POINT (4.84401 52.36812),F86,Overtoomse Veld,1012,1129,0.564216,571.0
...,...,...,...,...,...,...,...,...,...,...
386,ChIJVfi2HMgJxkcRYcxaRHoFaeo,1,Hotel Season Star,POINT (4.89623 52.378),A01,Burgwallen-Nieuwe Zijde,8645,5140,0.000195,2.0
387,ChIJyZWa5boJxkcRoi_aC8NY2AM,1,SWEETS hotel Amstelschutsluis,POINT (4.90246 52.3621),A07,De Weteringschans,2259,1673,0.000598,1.0
388,ChIJf-ETabkJxkcRnjwYhBCoeos,1,Studioboom,POINT (4.90129 52.37326),A04,Nieuwmarkt/Lastage,3574,2191,0.000456,2.0
389,ChIJh5sG0ecJxkcRBNdP2z2Wv_o,1,Hotel Leidsegracht,POINT (4.88042 52.36512),A07,De Weteringschans,2259,1673,0.000598,1.0


In [15]:
hotels_wijk_gdf_belt = hotels_wijk_gdf[hotels_wijk_gdf["Wijkcode"].isin(["E36", "E37", "E38", "E39", "E41", "F11", "F86", "F87", "F88", "F89", 
                                                                         "K23", "K44", "K48", "K52", "K90", "T92", "K91", "M58", "T93"])]
hotels_wijk_gdf_belt = hotels_wijk_gdf_belt.groupby(["Wijkcode", "Wijknaam_x"])["Aantal hotelbedden"].mean().reset_index()

In [16]:
print(round(hotels_wijk_gdf_belt["Aantal hotelbedden"].sum() / hotelbeds["Aantal hotelbedden"].sum() * 100), str("percent"))

31 percent


In [31]:
pd.read_csv("C:/Users/isamu/OneDrive/THESIS/DATA/Google/P2 Data/times.csv")

,place_id,name,reviews,rating,main_category,categories,closed_on,review_keywords,price_range,reviews_per_rating,coordinates,hours,popular_times,query
0,ChIJFTdsYMMJxkcRryoUM7EyJY0,Nooch,795,4.4,Aziatisch fusion-restaurant,"[""Aziatisch fusion-restaurant"",""Aziatisch rest...","[""maandag""]","[{""keyword"":""pad thai"",""count"":37},{""keyword"":...",$$$$$$$,"{""1"":31,""2"":20,""3"":55,""4"":169,""5"":520}","{""latitude"":52.3723397,""longitude"":4.8840769}","[{""day"":""maandag"",""times"":[""Gesloten""]},{""day""...","{""Tuesday"":[{""hour_of_day"":6,""time_label"":""06:...",restaurants in negen straatjes
1,ChIJz09EBcMJxkcRwRi15EQTPog,Restaurant de Struisvogel,572,4.7,Restaurant,"[""Restaurant""]",Open All Days,"[{""keyword"":""gangen"",""count"":34},{""keyword"":""v...",$$$$$$$,"{""1"":5,""2"":7,""3"":23,""4"":106,""5"":431}","{""latitude"":52.370449199999996,""longitude"":4.8...","[{""day"":""maandag"",""times"":[""17:30-00:00""]},{""d...",Not Present,restaurants in negen straatjes
2,ChIJIYdU8sMJxkcRD1Ys8SczZTE,Lotti's,848,3.9,Restaurant,"[""Restaurant""]",Open All Days,"[{""keyword"":""atmosfeer"",""count"":84},{""keyword""...",$$,"{""1"":69,""2"":74,""3"":89,""4"":216,""5"":400}","{""latitude"":52.371868899999996,""longitude"":4.8...","[{""day"":""maandag"",""times"":[""07:00-00:00""]},{""d...",Not Present,restaurants in negen straatjes
3,ChIJMZGUGcMJxkcRPRyYTICSTYM,Restaurant 't Zwaantje,2082,4.5,Nederlands restaurant,"[""Nederlands restaurant"",""Europees restaurant""...",Open All Days,"[{""keyword"":""stoofpotje"",""count"":135},{""keywor...",$$$$$$$,"{""1"":69,""2"":35,""3"":110,""4"":391,""5"":1477}","{""latitude"":52.3704694,""longitude"":4.8838778}","[{""day"":""maandag"",""times"":[""16:30-22:30""]},{""d...","{""Monday"":[{""hour_of_day"":6,""time_label"":""06:0...",restaurants in negen straatjes
4,ChIJy3zD9cIJxkcRxfIlpjk47JQ,Bar Brasserie OCCO,248,4.5,Brasserie,"[""Brasserie"",""Bar"",""Cocktailbar"",""Restaurant""]",Open All Days,"[{""keyword"":""cocktails"",""count"":24},{""keyword""...",NaN,"{""1"":9,""2"":3,""3"":20,""4"":37,""5"":179}","{""latitude"":52.369290299999996,""longitude"":4.8...","[{""day"":""maandag"",""times"":[""07:00-00:00""]},{""d...","{""Monday"":[{""hour_of_day"":6,""time_label"":""06:0...",restaurants in negen straatjes
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7325,ChIJPcEtxHEKxkcRx84qVvhT_uE,Cityden Zuidas,1159,4.3,Hotel,"[""Hotel"",""Bar"",""Hotel voor langdurig verblijf""...",Open All Days,"[{""keyword"":""tram"",""count"":68},{""keyword"":""ope...",NaN,"{""1"":52,""2"":29,""3"":90,""4"":310,""5"":678}","{""latitude"":52.3169719,""longitude"":4.8717681}",[],Not Present,hotels in zuidas
7326,ChIJVWZIVwsKxkcR4vauw0vYxuk,"Holiday Inn Express Amsterdam - South, an IHG ...",1125,4.3,Hotel,"[""Hotel"",""Congrescentrum"",""Trouwlocatie""]",Open All Days,"[{""keyword"":""tram"",""count"":44},{""keyword"":""ope...",NaN,"{""1"":37,""2"":27,""3"":127,""4"":345,""5"":589}","{""latitude"":52.325753999999996,""longitude"":4.8...",[],Not Present,hotels in zuidas
7327,ChIJC_KvV9jhxUcRgDzDAbDHkBw,Amsterdam Forest Hotel,766,4.2,Hotel,"[""Hotel"",""Accomodatie""]",Open All Days,"[{""keyword"":""douche"",""count"":32},{""keyword"":""o...",NaN,"{""1"":44,""2"":24,""3"":89,""4"":216,""5"":393}","{""latitude"":52.317434,""longitude"":4.8566474}",[],Not Present,hotels in zuidas
7328,ChIJN8RMMqgLxkcRknxrSHEle1E,Aparthotel Adagio Amsterdam City South,759,4.5,Hotel,"[""Hotel""]",Open All Days,"[{""keyword"":""tramhalte"",""count"":22},{""keyword""...",NaN,"{""1"":29,""2"":10,""3"":36,""4"":158,""5"":526}","{""latitude"":52.3184215,""longitude"":4.870786799...",[],Not Present,hotels in zuidas


In [11]:
#hotels_wijk_gdf.to_csv("C:/Users/isamu/OneDrive/THESIS/DATA/Amsterdam/hotelbed_hotel.csv")

In [33]:
hotels_wijk_gdf[hotels_wijk_gdf["Wijkcode"].isin(["E36", "E37", "E38", "E39", "E41", "F11", "F86", "F87", "F88", "F89", 
                                                  "K23", "K44", "K48", "K52", "K90", "T92", "K91", "M58", "T93"])]

,place_id,reviews 2024,name,geometry,Wijkcode,Wijknaam_x,Aantal hotelbedden,reviews 2024_sum,percentage_reviews_wijk,hotelbeds hotel
0,ChIJbeLDaSIKxkcR4uMOfG8J_cw,784,Hotel nhow Amsterdam RAI,POINT (4.89191 52.33896),K23,Zuidas,3129,1989,0.394168,1233.0
2,ChIJgYCiu5kLxkcRPQIZ1Wezozs,722,"Holiday Inn Express Amsterdam - Arena Towers, ...",POINT (4.94102 52.3096),T92,Amstel III/Bullewijk,4695,3196,0.225907,1061.0
3,ChIJxYYOH_bixUcRmvXuhmyjUx4,709,MEININGER Hotel Amsterdam City West,POINT (4.83746 52.39036),F11,Bedrijventerrein Sloterdijk,4705,2963,0.239285,1126.0
4,ChIJhfOaePYJxkcRWWJAZeXdlp4,637,Leonardo Hotel Amsterdam Rembrandtpark,POINT (4.84401 52.36812),F86,Overtoomse Veld,1012,1129,0.564216,571.0
5,ChIJ-RAyRD0KxkcRG4koK51opDE,631,Hotel Novotel Amsterdam City,POINT (4.88852 52.33372),K91,Buitenveldert-Oost,1646,637,0.990581,1630.0
6,ChIJ_WZ4cs0LxkcR8gFUHvfuBsE,629,Leonardo Royal Hotel Amsterdam,POINT (4.91785 52.33243),M58,Omval/Overamstel,3353,2097,0.299952,1006.0
7,ChIJv6OOxGDixUcR0NBBnqvC4sw,629,XO Hotels Park West,POINT (4.84706 52.3856),E36,Sloterdijk,817,938,0.670576,548.0
11,ChIJJVSNLlnixUcRa8wszX39t7M,546,Holiday Inn Express Amsterdam - Sloterdijk Sta...,POINT (4.83756 52.38783),F11,Bedrijventerrein Sloterdijk,4705,2963,0.184273,867.0
14,ChIJ2bm1R4MLxkcRO3Nce2k7SRE,527,Hotel Levell Amsterdam,POINT (4.95053 52.30475),T92,Amstel III/Bullewijk,4695,3196,0.164894,774.0
18,ChIJK_RWo1XhxUcRQlaKrcBHkNI,513,Olympic Hotel Amsterdam,POINT (4.85309 52.34136),K48,Stadionbuurt,732,580,0.884483,647.0


In [ ]:
#LOOK AT HOTELS WITH HIGH REVIEWS LOW BEDS AND FIX